# Exercise 68 — Bulk loading patterns

## Concept

For very large loads, even batched `INSERT` is slow because every row goes through SQL parsing and a row-level insert path. Production databases offer a bulk-copy path that bypasses both — Postgres calls it `COPY`.

### Postgres: `COPY ... FROM STDIN`

With psycopg2:

```python
with open("orders.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        "COPY orders (order_id, customer, amount, paid) FROM STDIN WITH CSV HEADER",
        f,
    )
conn.commit()
```

Or from an in-memory buffer (e.g., when transforming on the fly):

```python
import io
buf = io.StringIO()
for row in rows:
    buf.write(f"{row.id}\t{row.name}\t{row.amount}\n")
buf.seek(0)
cur.copy_from(buf, "orders", columns=("id", "name", "amount"))
```

Typical speedup over `executemany`: **10–100×**.

### SQLite: no native bulk-loader, but transactions matter

SQLite doesn't have `COPY`. The two things that move the needle for it:
1. **One transaction for the whole load** (one `commit` at the end, not per row).
2. **Drop/disable indexes during load**, recreate after.

Both covered below.

### General principle

Most bulk-load patterns share the same shape:
1. **Prepare the destination** — table exists, indexes possibly dropped.
2. **Stream the source** — never materialize the whole thing in memory.
3. **Single transaction** — atomic, no partial state.
4. **Recreate indexes** at the end if you dropped them.
5. **VACUUM / ANALYZE** so the planner has fresh stats (Postgres `ANALYZE`, SQLite `ANALYZE`).

## Setup — fixture with ~5000 rows

In [ ]:
from pathlib import Path

lines = ["order_id,customer,amount,paid\n"]
for i in range(1, 5001):
    lines.append(f"{i},Customer_{i % 50},{(i % 100) + 0.5},{1 if i % 3 else 0}\n")

CSV = "big_orders.csv"
Path(CSV).write_text("".join(lines), encoding="utf-8")
print("wrote", CSV, " rows:", len(lines) - 1)


## Task 1 — Single-transaction bulk load (SQLite)

Write a function that loads the CSV into SQLite **in a single transaction**: one big `executemany`, one `commit`, no per-row overhead.

```python
def bulk_load_sqlite(csv_path: str, db_path: str) -> int:
    """Load all rows in one transaction. Return rows inserted."""
```

Create the `orders` table (drop first to start fresh). Insert all rows via `executemany`. One commit at the end. Verify the row count.

In [ ]:
import csv
import sqlite3
from pathlib import Path

DB = "bulk.db"
Path(DB).unlink(missing_ok=True)

def bulk_load_sqlite(csv_path: str, db_path: str) -> int:
    pass  # TODO

n = bulk_load_sqlite(CSV, DB)
assert n == 5000
with sqlite3.connect(DB) as conn:
    count = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
assert count == 5000
print("ok")


## Task 2 — Compare per-row vs batched timings

Write a function that loads the same CSV three ways and returns a dict of elapsed seconds. Use a small subset (first 1000 rows) so the slow version doesn't take forever.

```python
def compare_load_strategies(csv_path: str, n_rows: int) -> dict[str, float]:
    """Returns {
        'per_row_commit':       seconds for execute()+commit() per row,
        'per_row_no_commit':    seconds for execute() per row, single commit at end,
        'executemany':          seconds for executemany() of the whole batch,
    }
    """
```

Use a fresh in-memory DB for each strategy (`sqlite3.connect(":memory:")`). Use `time.perf_counter()`.

The assertion just verifies the **ordering** — `executemany` should be the fastest. We don't assert hard times because they're machine-dependent.

In [ ]:
import csv
import sqlite3
import time

def compare_load_strategies(csv_path: str, n_rows: int) -> dict[str, float]:
    pass  # TODO

timings = compare_load_strategies(CSV, 1000)
print(timings)
assert timings["executemany"] < timings["per_row_no_commit"] < timings["per_row_commit"]
print("ok")


## Task 3 — Postgres COPY pattern (using mocks)

Real Postgres connection is not assumed. Build the **shape** of a `COPY FROM STDIN` loader using a stub that records the call.

```python
class StubPgCursor:
    def __init__(self):
        self.copy_calls = []     # list of (sql, file_contents_first_line)
    def copy_expert(self, sql: str, file) -> None:
        head = file.readline()
        self.copy_calls.append((sql, head))
    def __enter__(self): return self
    def __exit__(self, *a): return False

class StubPgConn:
    def __init__(self): self.cur = StubPgCursor(); self.commits = 0
    def cursor(self): return self.cur
    def commit(self): self.commits += 1
    def close(self): pass
```

Write a function that uses `copy_expert` to load a CSV file into a Postgres `orders` table with header row. Commit after.

```python
def pg_copy_from_csv(conn, csv_path: str, table: str = "orders") -> None:
    ...
```

Verify the recorded `copy_expert` call has:
- A SQL string starting with `COPY <table> FROM STDIN` and containing `CSV HEADER`
- The file's first line is the header row from the CSV

In [ ]:
class StubPgCursor:
    def __init__(self):
        self.copy_calls = []
    def copy_expert(self, sql, file):
        head = file.readline()
        self.copy_calls.append((sql, head))
    def __enter__(self): return self
    def __exit__(self, *a): return False

class StubPgConn:
    def __init__(self):
        self.cur = StubPgCursor()
        self.commits = 0
    def cursor(self): return self.cur
    def commit(self): self.commits += 1
    def close(self): pass

def pg_copy_from_csv(conn, csv_path: str, table: str = "orders") -> None:
    pass  # TODO

conn = StubPgConn()
pg_copy_from_csv(conn, CSV, "orders")

assert len(conn.cur.copy_calls) == 1
sql, head = conn.cur.copy_calls[0]
assert sql.startswith("COPY orders")
assert "FROM STDIN" in sql
assert "CSV HEADER" in sql
assert head.strip() == "order_id,customer,amount,paid"
assert conn.commits == 1
print("ok")
